# Experiment 3: More Inference Steps (T=400) â€” ResNet-18 CIFAR-10

**Hypothesis**: With 200 inference steps, the iterative ALM inference may not fully
converge for a 31-node graph. Doubling to T=400 gives error signals more time to
propagate through all 18 layers, potentially improving equilibrium quality and accuracy.

**Change vs baseline**: `infer_steps = 400` (was 200). Everything else identical.
Expected to roughly double ALM training time (~8h/trial).

**Baseline result**: ALM 57.6% vs Backprop 80.4% (T=200, eta=0.05, alpha=0.5)

## Imports & Setup

In [1]:
import jax
import numpy as np
import optax
import time

from fabricpc.nodes import ConvNode, Linear, IdentityNode, SkipConnection, AvgPool
from fabricpc.core.topology import Edge
from fabricpc.graph_assembly import TaskMap, graph
from fabricpc.core.inference import InferenceALM
from fabricpc.graph_initialization import initialize_params
from fabricpc.core.activations import (
    IdentityActivation,
    ReLUActivation,
    TanhActivation,
    GeluActivation,
    LeakyReLUActivation,
    SoftmaxActivation,
)
from fabricpc.core.energy import CrossEntropyEnergy
from fabricpc.core.initializers import XavierInitializer
from fabricpc.training import train_pcn, evaluate_pcn
from fabricpc.training.train_backprop import train_backprop, evaluate_backprop
from fabricpc.experiments import ExperimentArm, ABExperiment
from fabricpc.utils.data.dataloader import Cifar10Loader
from fabricpc import setup_jax

setup_jax()
jax.config.update("jax_default_prng_impl", "threefry2x32")

/home/shamir/jax-cuda-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Helper Classes & Functions

In [2]:
def get_activation(name):
    factories = {
        "relu": ReLUActivation,
        "tanh": TanhActivation,
        "gelu": GeluActivation,
        "leaky_relu": lambda: LeakyReLUActivation(alpha=0.1),
    }
    if name not in factories:
        raise ValueError(f"Unknown activation: {name}. Choose from {list(factories)}")
    return factories[name]()


class AugmentedCifar10Loader:
    """Wraps Cifar10Loader with random horizontal flip and random crop+pad."""

    def __init__(self, base_loader, seed=42, pad=4):
        self.base_loader = base_loader
        self.seed = seed
        self.pad = pad
        self._epoch = 0

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self._epoch)
        self._epoch += 1
        pad = self.pad
        for images, labels in self.base_loader:
            flip_mask = rng.random(images.shape[0]) > 0.5
            images[flip_mask] = images[flip_mask, :, ::-1, :]

            padded = np.pad(
                images, ((0, 0), (pad, pad), (pad, pad), (0, 0)), mode="reflect"
            )
            B, H, W, C = images.shape
            crop_y = rng.integers(0, 2 * pad + 1, size=B)
            crop_x = rng.integers(0, 2 * pad + 1, size=B)
            for i in range(B):
                images[i] = padded[
                    i, crop_y[i] : crop_y[i] + H, crop_x[i] : crop_x[i] + W, :
                ]

            yield images, labels

    def __len__(self):
        return len(self.base_loader)

## ResNet-18 Graph Builder

In [3]:
def make_residual_block(
    prev_node, channels, stride, block_name, weight_init,
    activation=ReLUActivation(),
):
    """Create one residual block: conv_a -> conv_b(act) -> skip(sum)."""
    in_h, in_w, in_channels = prev_node._shape
    out_h = in_h // stride if stride > 1 else in_h
    out_w = in_w // stride if stride > 1 else in_w

    conv_a = ConvNode(
        shape=(out_h, out_w, channels), kernel_size=(3, 3),
        stride=(stride, stride), padding="SAME",
        activation=activation, weight_init=weight_init,
        name=f"{block_name}_conv_a",
    )
    conv_b = ConvNode(
        shape=(out_h, out_w, channels), kernel_size=(3, 3),
        stride=(1, 1), padding="SAME",
        activation=activation, weight_init=weight_init,
        name=f"{block_name}_conv_b",
    )
    skip_node = SkipConnection(shape=(out_h, out_w, channels), name=f"{block_name}_skip_sum")

    nodes = [conv_a, conv_b, skip_node]
    edges = [
        Edge(source=prev_node, target=conv_a.slot("in")),
        Edge(source=conv_a, target=conv_b.slot("in")),
        Edge(source=conv_b, target=skip_node.slot("in")),
    ]

    needs_downsample = (stride != 1) or (in_channels != channels)
    if needs_downsample:
        conv_skip = ConvNode(
            shape=(out_h, out_w, channels), kernel_size=(1, 1),
            stride=(stride, stride), padding="SAME",
            activation=IdentityActivation(), weight_init=weight_init,
            name=f"{block_name}_skip",
        )
        nodes.append(conv_skip)
        edges.append(Edge(source=prev_node, target=conv_skip.slot("in")))
        edges.append(Edge(source=conv_skip, target=skip_node.slot("skip")))
    else:
        edges.append(Edge(source=prev_node, target=skip_node.slot("skip")))

    return nodes, edges, skip_node


def build_resnet18(
    weight_init, activation=ReLUActivation(), *,
    infer_steps, eta_infer, alpha=1.0, rho=1.0,
):
    """Build ResNet-18 [2,2,2,2] for CIFAR-10 with InferenceALM."""
    input_node = IdentityNode(shape=(32, 32, 3), name="input")
    stem = ConvNode(
        shape=(32, 32, 32), kernel_size=(3, 3), stride=(1, 1), padding="SAME",
        activation=activation, weight_init=weight_init, name="stem",
    )

    all_nodes = [input_node, stem]
    all_edges = [Edge(source=input_node, target=stem.slot("in"))]

    stage_configs = [(32, 1, 2), (64, 2, 2), (128, 2, 2), (256, 2, 2)]
    prev = stem
    for stage_idx, (channels, first_stride, num_blocks) in enumerate(stage_configs, 1):
        for block_idx in range(num_blocks):
            stride = first_stride if block_idx == 0 else 1
            nodes, edges, prev = make_residual_block(
                prev, channels, stride, f"s{stage_idx}b{block_idx + 1}",
                weight_init, activation,
            )
            all_nodes.extend(nodes)
            all_edges.extend(edges)

    avg_pool = AvgPool(shape=(256,), name="avgpool", global_pool=True)
    all_nodes.append(avg_pool)
    all_edges.append(Edge(source=prev, target=avg_pool.slot("in")))

    output = Linear(
        shape=(10,), activation=SoftmaxActivation(), energy=CrossEntropyEnergy(),
        flatten_input=True, weight_init=XavierInitializer(), name="output",
    )
    all_nodes.append(output)
    all_edges.append(Edge(source=avg_pool, target=output.slot("in")))

    structure = graph(
        nodes=all_nodes, edges=all_edges,
        task_map=TaskMap(x=input_node, y=output),
        inference=InferenceALM(
            eta_infer=eta_infer, infer_steps=infer_steps,
            alpha=alpha, rho=rho,
        ),
    )
    return structure

## Configuration

In [4]:
# --- Experiment settings ---
n_trials = 2
verbose = False

# --- Training hyperparameters ---
num_epochs = 30
batch_size = 128
lr = 0.001
weight_decay = 0.01
augment = False

# --- Activations (per arm) ---
alm_activation_name = "tanh"
bp_activation_name  = "relu"

# --- ALM inference settings ---
infer_steps = 400   # KEY CHANGE: doubled from baseline 200
eta_infer   = 0.05
alm_alpha   = 0.5
alm_rho     = 1.0

train_config = {"num_epochs": num_epochs}

## Model Factories

In [5]:
def create_alm_model(rng_key):
    """ALM arm â€” Xavier init, T=400 inference steps."""
    structure = build_resnet18(
        weight_init=XavierInitializer(),
        activation=get_activation(alm_activation_name),
        infer_steps=infer_steps, eta_infer=eta_infer,
        alpha=alm_alpha, rho=alm_rho,
    )
    params = initialize_params(structure, rng_key)
    return params, structure


def create_backprop_model(rng_key):
    """Backprop arm â€” standard Xavier init."""
    structure = build_resnet18(
        weight_init=XavierInitializer(),
        activation=get_activation(bp_activation_name),
        infer_steps=infer_steps, eta_infer=eta_infer,
        alpha=alm_alpha, rho=alm_rho,
    )
    params = initialize_params(structure, rng_key)
    return params, structure


_probe_params, _probe_structure = create_backprop_model(jax.random.PRNGKey(0))
print(f"Model: {len(_probe_structure.nodes)} nodes, {len(_probe_structure.edges)} edges")
total_params = sum(p.size for p in jax.tree_util.tree_leaves(_probe_params))
print(f"Total parameters: {total_params:,}")
del _probe_params, _probe_structure

Model: 31 nodes, 38 edges
Total parameters: 2,795,210


## Run Experiment

In [6]:
print("=" * 70)
print("EXPERIMENT 3: More Inference Steps (T=400)")
print("ResNet-18 on CIFAR-10: ALM (T=400) vs Backpropagation")
print("=" * 70)
print(
    "Architecture: stem(32) -> s1[2x32] -> s2[2x64] -> s3[2x128] "
    "-> s4[2x256] -> avgpool -> 10"
)
print(
    f"ALM: {alm_activation_name} + Xavier, {infer_steps} steps "
    f"@ eta={eta_infer}, alpha={alm_alpha}, rho={alm_rho}  |  "
    f"Backprop: {bp_activation_name} + Xavier"
)
print(
    f"Epochs: {num_epochs}  |  Batch size: {batch_size}  |  LR: {lr}  "
    f"|  Augment: {augment}"
)
print(f"Trials: {n_trials}")
print()

steps_per_epoch = len(
    Cifar10Loader("train", batch_size=batch_size, shuffle=True, seed=0)
)
total_steps = num_epochs * steps_per_epoch
warmup_steps = int(0.05 * total_steps)

schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0, peak_value=lr,
    warmup_steps=warmup_steps, decay_steps=total_steps, end_value=lr * 0.01,
)
optimizer = optax.adamw(schedule, weight_decay=weight_decay)

arm_alm = ExperimentArm(
    name="ALM-T400",
    model_factory=create_alm_model,
    train_fn=train_pcn,
    eval_fn=evaluate_pcn,
    optimizer=optimizer,
    train_config=train_config,
)

arm_bp = ExperimentArm(
    name="Backprop",
    model_factory=create_backprop_model,
    train_fn=train_backprop,
    eval_fn=evaluate_backprop,
    optimizer=optimizer,
    train_config=train_config,
)


def data_loader_factory(seed):
    base_train = Cifar10Loader("train", batch_size=batch_size, shuffle=True, seed=seed)
    train_loader = AugmentedCifar10Loader(base_train, seed=seed) if augment else base_train
    test_loader = Cifar10Loader("test", batch_size=batch_size, shuffle=False)
    return train_loader, test_loader


experiment = ABExperiment(
    arm_a=arm_alm, arm_b=arm_bp,
    metric="accuracy",
    data_loader_factory=data_loader_factory,
    n_trials=n_trials, verbose=verbose,
)

results = experiment.run()
results.print_summary()

EXPERIMENT 3: More Inference Steps (T=400)
ResNet-18 on CIFAR-10: ALM (T=400) vs Backpropagation
Architecture: stem(32) -> s1[2x32] -> s2[2x64] -> s3[2x128] -> s4[2x256] -> avgpool -> 10
ALM: tanh + Xavier, 400 steps @ eta=0.05, alpha=0.5, rho=1.0  |  Backprop: relu + Xavier
Epochs: 30  |  Batch size: 128  |  LR: 0.001  |  Augment: False
Trials: 2

--- Trial 1/2 (seed=0) ---


Epoch 30/30: 100%|██████████| 11730/11730 [7:13:21<00:00,  2.22s/it, energy=0.5265, epoch=30/30]  


  ALM-T400: accuracy=0.5531  (train: 26002.0s)
  Backprop: accuracy=0.8042  (train: 230.6s)
--- Trial 2/2 (seed=1000) ---


Epoch 14/30:  45%|████▌     | 5289/11730 [3:15:52<4:02:29,  2.26s/it, energy=2.5309, epoch=14/30]

KeyboardInterrupt: 